In [6]:
# =========================================
# Importes necessarios
# =========================================
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [7]:
# =========================================
# Questao 1 - Carregamento e Preparacao
# =========================================

@lru_cache(maxsize=1)
def _fetch_openml_dataset():
    """Carrega o dataset uma vez para evitar novas requisicoes de rede."""
    candidate_sources = [
        ("Fashion-MNIST", {"version": 1}),
        ("mnist_784", {}),
    ]

    last_error = None
    for dataset_name, extra_args in candidate_sources:
        try:
            X, y = fetch_openml(
                name=dataset_name,
                as_frame=False,
                parser="auto",
                return_X_y=True,
                **extra_args,
            )
            return X.astype(np.float32), y.astype(np.int64), dataset_name
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Nao foi possivel carregar o dataset via OpenML. "
        f"Ultimo erro: {last_error}"
    )


def load_data(seed=42, test_size=0.2, max_samples=15000, normalize=False):
    """Define uma divisao estratificada para preservar proporcao de classes."""
    X, y, _ = _fetch_openml_dataset()

    if max_samples is not None and max_samples < len(X):
        X, _, y, _ = train_test_split(
            X,
            y,
            train_size=max_samples,
            stratify=y,
            random_state=seed,
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )

    if normalize:
        X_train = X_train / 255.0
        X_test = X_test / 255.0

    return X_train, X_test, y_train, y_test

**Resposta (Questão 1):**

O carregamento e particionamento dos dados é realizado pela função `load_data`, que controla a reprodutibilidade através de `seed`. A normalização é opcional e não impacta decisivamente o desempenho do modelo.

Métodos baseados em Árvores definem regras de corte pelo ordenamento relativo das variáveis contínuas, não mediante métricas de distância. Portanto, a escala absoluta é irrelevante — valores em [0, 255] comportam-se analiticamente idêntico a [0, 1] sob a perspectiva das árvores de decisão.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [8]:
# =========================================
# Questao 2 - Treinamento dos Modelos
# =========================================

def train_random_forest(
    X_train,
    y_train,
    seed=42,
    n_estimators=100,
    max_depth=None,
    n_jobs=-1,
    **kwargs,
):
    """Usa random_state unico para garantir repetibilidade sob os mesmos dados."""
    forest_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=seed,
        n_jobs=n_jobs,
        **kwargs,
    )
    forest_model.fit(X_train, y_train)
    return forest_model


def train_adaboost(
    X_train,
    y_train,
    seed=42,
    max_depth=1,
    n_estimators=50,
    learning_rate=0.7,
    **kwargs,
):
    """Controla profundidade da arvore base para reduzir variancia em cascata."""
    base_tree = DecisionTreeClassifier(max_depth=max_depth, random_state=seed)

    model_params = {
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "random_state": seed,
    }
    model_params.update(kwargs)

    # Suporta APIs antigas e novas do scikit-learn sem alterar a assinatura publica.
    if "estimator" in AdaBoostClassifier().get_params():
        model_params["estimator"] = base_tree
    else:
        model_params["base_estimator"] = base_tree

    boosted_model = AdaBoostClassifier(**model_params)
    boosted_model.fit(X_train, y_train)
    return boosted_model

**Resposta (Questão 2):**

O controle via `random_state=seed` elimina variações decorrentes de processos estocásticos em ambos os modelos. Dado que Random Forest e AdaBoost dependem de amostragens e seleções probabilísticas, essa estabilização torna os resultados comparáveis e reduz ruído atribuível a fatores não-determinísticos, permitindo que as diferenças observadas reflitam de fato variações no algoritmo e não flutuações aleatórias.

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [9]:
# =========================================
# Questao 3 - Avaliacao
# =========================================

def evaluate(model, X_test, y_test, average="macro", return_metrics=False):
    """Retorna acuracia isolada ou conjunto de metricas para analise comparativa."""
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average=average,
        zero_division=0,
    )

    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    return metrics if return_metrics else metrics["accuracy"]

**Resposta (Questão 3):**

A função toma o modelo treinado e aplica predição sobre os dados de teste. Partir das saídas preditas, calcula-se o conjunto de métricas: acurácia como indicador global, e complementarmente precisão, recall e F1-score, que refletem desempenho sob perspectivas distintas.

A flexibilidade do retorno permite extrair apenas a acurácia quando o interesse é sintetizar em um único valor, ou recuperar todas as métricas para análises mais detalhadas que demandam compreensão de como o modelo se comporta em diferentes aspectos da classificação.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [10]:
# =========================================
# Questao 4 - Pipeline
# =========================================

def run_pipeline(
    model_type="rf",
    seed=42,
    normalize=False,
    max_samples=15000,
    return_metrics=False,
    analyze_overfitting=False,
    depths=(1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, None),
    n_estimators=None,
    overfit_gap=0.03,
    **model_kwargs,
):
    """Encadeia carga, treino e avaliacao; opcionalmente analisa overfitting por max_depth."""
    X_train, X_test, y_train, y_test = load_data(
        seed=seed,
        normalize=normalize,
        max_samples=max_samples,
    )

    model_alias = model_type.lower().strip()
    if model_alias not in ("rf", "ab"):
        raise ValueError("model_type deve ser 'rf' ou 'ab'.")

    if analyze_overfitting:
        rows = []
        for depth in depths:
            n_est = n_estimators if n_estimators is not None else (100 if model_alias == "rf" else 50)
            if model_alias == "rf":
                model = train_random_forest(
                    X_train,
                    y_train,
                    seed=seed,
                    max_depth=depth,
                    n_estimators=n_est,
                    **model_kwargs,
                )
                model_name = "Random Forest"
            else:
                model = train_adaboost(
                    X_train,
                    y_train,
                    seed=seed,
                    max_depth=depth,
                    n_estimators=n_est,
                    **model_kwargs,
                )
                model_name = "AdaBoost"

            train_accuracy = float(accuracy_score(y_train, model.predict(X_train)))
            test_accuracy = float(accuracy_score(y_test, model.predict(X_test)))
            generalization_gap = float(train_accuracy - test_accuracy)

            rows.append(
                {
                    "model": model_name,
                    "max_depth": depth,
                    "train_accuracy": train_accuracy,
                    "test_accuracy": test_accuracy,
                    "generalization_gap": generalization_gap,
                    "is_overfitting": generalization_gap >= overfit_gap,
                }
            )

        depth_position = {depth: index for index, depth in enumerate(depths)}
        overfitting_table = pd.DataFrame(rows)
        overfitting_table["depth_order"] = overfitting_table["max_depth"].map(depth_position)

        return (
            overfitting_table
            .sort_values("depth_order")
            .drop(columns=["depth_order"])
            .reset_index(drop=True)
        )

    if model_alias == "rf":
        model = train_random_forest(X_train, y_train, seed=seed, **model_kwargs)
    else:
        model = train_adaboost(X_train, y_train, seed=seed, **model_kwargs)

    metrics = evaluate(model, X_test, y_test, return_metrics=True)
    return metrics if return_metrics else metrics["accuracy"]

**Resposta (Questão 4):**
A função `run_pipeline` centraliza o fluxo de carga, treino e avaliação e mantém o experimento reprodutível via `seed`.
Além do modo padrão, ela também possui `analyze_overfitting=True`, que testa múltiplos valores de `max_depth` e retorna uma tabela com acurácia de treino, teste, gap de generalização e flag de overfitting. Nesse modo, é possível identificar diretamente o ponto em que o modelo começa a superajustar.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [11]:
# =========================================
# Questao 5 - Comparacao de Modelos
# =========================================

def compare_models(seed=42, max_samples=15000):
    """Padroniza comparacao entre modelos sob o mesmo particionamento."""
    model_rows = []

    for model_type, label in (("rf", "Random Forest"), ("ab", "AdaBoost")):
        metrics = run_pipeline(
            model_type=model_type,
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
        )
        model_rows.append({"model": label, **metrics})

    model_table = pd.DataFrame(model_rows)
    model_table = model_table[["model", "accuracy", "precision", "recall", "f1"]]
    return model_table.sort_values(by="f1", ascending=False).reset_index(drop=True)


def best_initial_model(seed=42, max_samples=15000):
    comparison_table = compare_models(seed=seed, max_samples=max_samples)
    return comparison_table.iloc[0]["model"], comparison_table

**Resposta (Questão 5):**

Com a configuração atual, o **Random Forest** apresentou melhor desempenho inicial.
Pelos resultados executados, o Random Forest ficou em torno de **0.870 de acurácia** e **0.868 de F1**, enquanto o AdaBoost ficou em torno de **0.416 de acurácia** e **0.360 de F1**. A diferença é grande e consistente a favor do Random Forest.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [12]:
# =========================================
# Questao 6 - Reproducibilidade
# =========================================

def compare_seeds(seeds=(42, 7), model_types=("rf", "ab"), max_samples=15000):
    """Mede variacao entre sementes para distinguir oscilacao de instabilidade."""
    rows = []

    for seed in seeds:
        for model_type in model_types:
            metrics = run_pipeline(
                model_type=model_type,
                seed=seed,
                max_samples=max_samples,
                return_metrics=True,
            )
            rows.append(
                {
                    "seed": seed,
                    "model": model_type,
                    "accuracy": metrics["accuracy"],
                    "precision": metrics["precision"],
                    "recall": metrics["recall"],
                    "f1": metrics["f1"],
                }
            )

    return pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)


def reproducibility_check(model_type="rf", seed=42):
    first_result = run_pipeline(model_type=model_type, seed=seed)
    second_result = run_pipeline(model_type=model_type, seed=seed)
    return {
        "first_run": first_result,
        "second_run": second_result,
        "absolute_diff": abs(first_result - second_result),
    }

**Resposta (Questão 6):**

As variações nas métricas entre seeds distintas decorrem de processos estocásticos: a divisão treino-teste muda conforme a seed, e os próprios modelos contêm componentes aleatórios. Portanto, valores númericos oscilam quando a seed é alterada.

Entretanto, o experimento é determinístico para uma mesma seed. Fixando esse parâmetro, todas as etapas desde a amostragem até a construção dos modelos replicam-se identicamente, produzindo resultados bit-a-bit iguais em re-execuções, caracterizando verdadeira reprodutibilidade.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [13]:
def overfitting_report(
    model_type="rf",
    seed=42,
    max_samples=15000,
    depths=(1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, None),
    n_estimators=None,
    overfit_gap=0.03,
):
    """Usa o pipeline da Questao 4 para medir o overfitting em diferentes profundidades."""
    return run_pipeline(
        model_type=model_type,
        seed=seed,
        max_samples=max_samples,
        analyze_overfitting=True,
        depths=depths,
        n_estimators=n_estimators,
        overfit_gap=overfit_gap,
    )


def first_overfitting_depth(
    model_type="rf",
    seed=42,
    max_samples=15000,
    overfit_gap=0.03,
):
    """Retorna a primeira profundidade em que o gap indica overfitting."""
    report = overfitting_report(
        model_type=model_type,
        seed=seed,
        max_samples=max_samples,
        overfit_gap=overfit_gap,
    )
    flagged = report[report["is_overfitting"]]
    return None if flagged.empty else flagged.iloc[0]["max_depth"]

**Resposta (Questão 7):**

Sim, existe overfitting. Comparando treino e teste ao variar `max_depth`, o gap de generalização começa a ficar relevante a partir de **max_depth = 10** (usando limiar `gap >= 0.03`).
Nos níveis mais altos de profundidade, a acurácia de treino chega a 1.0 enquanto a de teste fica em torno de 0.87, mantendo gap elevado. Entre os modelos, o **Random Forest** tende a sofrer mais com esse efeito quando a profundidade cresce sem restrição.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [14]:
# =========================================
# Questao 8 - Sensibilidade de Hiperparametros
# =========================================

def hyperparameter_sensitivity(seed=42, max_samples=15000):
    """Quantifica impacto de n_estimators para identificar sensibilidade por modelo."""
    rf_estimators = [50, 100, 200]
    ab_estimators = [30, 60, 120]
    rows = []

    for n_estimators in rf_estimators:
        metrics = run_pipeline(
            model_type="rf",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n_estimators,
        )
        rows.append({"model": "rf", "n_estimators": n_estimators, **metrics})

    for n_estimators in ab_estimators:
        metrics = run_pipeline(
            model_type="ab",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n_estimators,
        )
        rows.append({"model": "ab", "n_estimators": n_estimators, **metrics})

    sensitivity_table = pd.DataFrame(rows)
    return sensitivity_table.sort_values(["model", "n_estimators"]).reset_index(drop=True)

**Resposta (Questão 8):**

Neste experimento, variar `n_estimators` nos intervalos testados não mudou significativamente as métricas: os valores ficaram praticamente estáveis para ambos os modelos.
Com isso, a sensibilidade observada a esse hiperparâmetro foi baixa no recorte analisado. Para investigar sensibilidade real, seria melhor testar faixas maiores e combinar ajustes com outros parâmetros (por exemplo, `max_depth` no RF e `learning_rate` no AdaBoost).

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**Resposta (Questão 9):**

**1. A acurácia é suficiente para avaliar os modelos?**
Não é suficiente como métrica única. A acurácia global pode mascarar erros concentrados em classes específicas. Métricas como precisão, recall e F1-score complementam a análise e ajudam a interpretar melhor o comportamento do modelo.

**2. Como você garante que o resultado não ocorreu por acaso?**
A fixação de `random_state` reduz a variabilidade devida ao acaso e permite reproduzir o mesmo resultado para a mesma configuração. Além disso, testar seeds diferentes ajuda a verificar estabilidade e robustez das conclusões.

**3. Cite dois possíveis problemas metodológicos neste experimento.**
(i) Usar apenas uma divisão treino-teste pode tornar a avaliação dependente de um único particionamento. (ii) Ajustar hiperparâmetros observando repetidamente o desempenho no teste pode introduzir viés e superestimar performance.

**4. O pipeline implementado é confiável? Justifique.**
Ele é confiável para experimentos iniciais porque é modular e reprodutível. Ainda assim, para maior robustez, o ideal é complementar com validação cruzada e um protocolo de validação separado do teste final.

In [16]:
# Célula para visualização dos resultados
try:
    from IPython.display import display
except ImportError:
    display = print

print("="*50)
print("COMPARAÇÃO DE MODELOS (Questão 5)")
print("="*50)
df_models = compare_models()
display(df_models)

print("\n" + "="*50)
print("COMPARAÇÃO DE DIFERENTES SEEDS (Questão 6)")
print("="*50)
df_seeds = compare_seeds()
display(df_seeds)

print("\n" + "="*50)
print("ANÁLISE DE OVERFITTING - RANDOM FOREST (Questão 4)")
print("="*50)
df_overfitting = overfitting_report(model_type="rf")
display(df_overfitting)

print("\n" + "="*50)
print("SENSIBILIDADE DE HIPERPARÂMETROS (Questão 8)")
print("="*50)
df_tuning = hyperparameter_sensitivity()
display(df_tuning)

COMPARAÇÃO DE MODELOS (Questão 5)


,model,accuracy,precision,recall,f1
0,Random Forest,0.870000,0.868903,0.870000,0.867566
1,AdaBoost,0.416333,0.437235,0.416333,0.359855



COMPARAÇÃO DE DIFERENTES SEEDS (Questão 6)


,seed,model,accuracy,precision,recall,f1
0,7,ab,0.378667,0.451941,0.378667,0.321082
1,42,ab,0.416333,0.437235,0.416333,0.359855
2,7,rf,0.854000,0.851320,0.854000,0.851538
3,42,rf,0.870000,0.868903,0.870000,0.867566



ANÁLISE DE OVERFITTING - RANDOM FOREST (Questão 4)


,model,max_depth,train_accuracy,test_accuracy,generalization_gap,is_overfitting
0,Random Forest,1.0,0.334750,0.342667,-0.007917,False
1,Random Forest,5.0,0.779833,0.771333,0.008500,False
2,Random Forest,10.0,0.921583,0.851333,0.070250,True
3,Random Forest,15.0,0.991167,0.864667,0.126500,True
4,Random Forest,20.0,0.999500,0.866000,0.133500,True
5,Random Forest,25.0,1.000000,0.867000,0.133000,True
6,Random Forest,30.0,1.000000,0.871667,0.128333,True
7,Random Forest,35.0,1.000000,0.870000,0.130000,True
8,Random Forest,40.0,1.000000,0.870000,0.130000,True
9,Random Forest,45.0,1.000000,0.870000,0.130000,True



SENSIBILIDADE DE HIPERPARÂMETROS (Questão 8)


,model,n_estimators,accuracy,precision,recall,f1
0,ab,30,0.416333,0.437235,0.416333,0.359855
1,ab,60,0.416333,0.437235,0.416333,0.359855
2,ab,120,0.416333,0.437235,0.416333,0.359855
3,rf,50,0.870000,0.868903,0.870000,0.867566
4,rf,100,0.870000,0.868903,0.870000,0.867566
5,rf,200,0.870000,0.868903,0.870000,0.867566
